In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import math


from geometryClass import gmshClass
from geometryClass import elmerClass
from geometryClass import geometryClass

In [2]:
inputParam = {
    'pitch': 100,
    'holeRadius': 25,
    'padLength': 44,
    'gridStandoff': 50,
    'cathodeHeight': 150,
    'gridThickness': 1,
    'dielectricThickness': 5,
    'pillarRadius': 6, 
    'driftField': 300,
    'fieldRatio': 50
}

testAll = geometryClass(inputParam)

In [3]:
testAll.createGeometry(runGUI=False)

In [5]:
testAll.solveEFields()

	Executing ElmerGrid...
	Executing ElmerSolver...
	Executing ElmerSolver for Weightings...


In [ ]:
cellParts = testGeometry.createFIMSGeometry(runGUI=True)

In [ ]:
fileName = testGeometry.createHexagonalGeometry(neighborCells=True, runGUI=True)

In [ ]:
testElmer = elmerClass(name=fileName, numPads=7, capacitance=True)

In [ ]:
testElmer.writeSIF()

In [ ]:
testElmer.runElmer()

In [ ]:
allScanResults = []

pitchValues = [30, 50, 80, 100]

for inPitch in pitchValues:

    padScanResults = []
    
    for inPad in np.linspace(2, round(inPitch*math.sqrt(3)/2), 10):
    
        inScanParam = {
            'pitch': inPitch,
            'holeRadius': 10,
            'padLength': inPad,
            'gridStandoff': 50,
            'cathodeHeight': 150,
            'gridThickness': 1,
            'dielectricThickness': 5
        }

        scanGeometry = gmshClass(inScanParam)
    
        scanFile = scanGeometry.createHexagonalGeometry(neighborCells=True, runGUI=False)
    
        scanElmer = elmerClass(scanFile, numPads=7, capacitance=True)
    
        scanElmer.writeSIF()
        scanElmer.runElmer()
    
        scanData = np.loadtxt('../Simulation/Geometry/elmerResults/capacitancematrix.dat')
        scanCapacitance = scanData*1e15
    
        centerPadData = scanCapacitance[3:, 2]
    
        averageCap = centerPadData.mean()
        sumCap = centerPadData.sum()
    
        gridCoupling = scanCapacitance[1, 2]
    
    
        scanResults = {
            'scanParam': inScanParam,
            'averageCap': centerPadData.mean(),
            'sumCap': centerPadData.sum(),
            'gridCoupling': scanCapacitance[1, 2],
            'secondOrder': (scanCapacitance[4, 3], scanCapacitance[4, 5])
        }
    
        padScanResults.append(scanResults)

    allScanResults.append(padScanResults)
 




In [ ]:
for inPitch in allScanResults:
    pitch = inPitch[0]['scanParam']['pitch']
    cellLength = pitch/math.sqrt(3)


    xVals = []
    yVals = []
    for inScan in inPitch:
        xVals.append(inScan['scanParam']['padLength']/cellLength)
        yVals.append(inScan['sumCap'])


    plt.scatter(
        xVals, yVals, 
        ls='', label=f'Pitch = {pitch} um'
    )

plt.xlabel('Pad Length / Cell Length')
plt.ylabel('Inter-Pad Capacitance (fF)')

plt.grid()
plt.legend()
#plt.yscale('log')

plt.show()


In [ ]:
print(allScanResults)

In [ ]:
pitchValues = [30, 50, 80]

for inPitch in pitchValues:    
    for inPad in range(5, round(inPitch/math.sqrt(3)), 5):


        print(f'{round(inPitch/math.sqrt(3))}, {inPad}')
        print(f'{inPad/round(inPitch/math.sqrt(3)):.3f}')